# Crypto 50x Leverage - Train & Backtest

## 중요: 기존 모델 사용 불가!

```
기존 CryptoEnv (41 dim) ≠ CryptoLeverageEnv (51 dim)
→ 반드시 새로 학습해야 함!
```

**노트북 구조**:
1. Import & Data Load
2. Environment 정의
3. **학습** ← 필수!
4. Backtest
5. Visualization

# Part 1. Import Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import A2C, PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback
import os

import gymnasium as gym
from gymnasium import spaces

# 레버리지 모델 저장 디렉토리 (기존 모델과 분리!)
LEVERAGE_MODEL_DIR = "./trained_models_leverage_50x"
os.makedirs(LEVERAGE_MODEL_DIR, exist_ok=True)

print("Packages loaded!")
print(f"Models will be saved to: {LEVERAGE_MODEL_DIR}")

Packages loaded!
Models will be saved to: ./trained_models_leverage_50x


# Part 2. Load Data

In [2]:
# Load data
data = np.load('crypto_5m_data.npz', allow_pickle=True)

train_price = data['train_price']
train_tech = data['train_tech']
test_price = data['test_price']
test_tech = data['test_tech']
crypto_pairs = data['crypto_pairs']

print(f"Train: {len(train_price)} samples")
print(f"Test: {len(test_price)} samples")
print(f"Coins: {len(crypto_pairs)} - {list(crypto_pairs)}")
print(f"Tech indicators: {train_tech.shape[1]}")

Train: 6823 samples
Test: 1706 samples
Coins: 5 - [np.str_('BTC/USDT'), np.str_('ETH/USDT'), np.str_('BNB/USDT'), np.str_('XRP/USDT'), np.str_('SOL/USDT')]
Tech indicators: 35


# Part 3. 50x Leverage Environment

In [3]:
class CryptoLeverageEnv:
    """
    50x Leverage Cryptocurrency Trading Environment
    
    State: [cash_ratio, positions(5), unrealized_pnl(5), tech_indicators(40)]
    Action: [-1, 1] per coin (positive=Long, negative=Short)
    
    Features:
    - Long/Short with 50x leverage
    - Liquidation at 1.5% loss (50x leverage)
    - Trailing stop loss (2%)
    - Risk-adjusted reward
    """
    
    def __init__(
        self,
        config,
        lookback=1,
        initial_capital=1e6,
        leverage=50,
        taker_fee=0.0004,
        maintenance_margin=0.005,
        trailing_stop_pct=0.02,
        use_trailing_stop=True,
        gamma=0.99,
        reward_scaling=1e-4,
        liquidation_penalty=10.0,
    ):
        self.lookback = lookback
        self.initial_capital = initial_capital
        self.leverage = leverage
        self.taker_fee = taker_fee
        self.maintenance_margin = maintenance_margin
        self.trailing_stop_pct = trailing_stop_pct
        self.use_trailing_stop = use_trailing_stop
        self.gamma = gamma
        self.reward_scaling = reward_scaling
        self.liquidation_penalty = liquidation_penalty
        
        self.price_array = config["price_array"]
        self.tech_array = config["tech_array"]
        
        self.crypto_num = self.price_array.shape[1]
        self.max_step = self.price_array.shape[0] - lookback - 1
        
        # State dimension
        self.state_dim = (
            1 +  # cash ratio
            self.crypto_num +  # positions
            self.crypto_num +  # unrealized PnL
            (self.price_array.shape[1] + self.tech_array.shape[1]) * lookback
        )
        self.action_dim = self.crypto_num
        self.if_discrete = False
        
        self.returns_history = []
        self.history = []
        self.reset()
    
    def reset(self, *, seed=None, options=None):
        self.time = self.lookback - 1
        self.cash = self.initial_capital
        self.current_price = self.price_array[self.time]
        self.current_tech = self.tech_array[self.time]
        
        self.positions = np.zeros(self.crypto_num, dtype=np.float32)
        self.entry_prices = np.zeros(self.crypto_num, dtype=np.float32)
        self.highest_prices = np.zeros(self.crypto_num, dtype=np.float32)
        self.lowest_prices = np.full(self.crypto_num, np.inf, dtype=np.float32)
        
        self.unrealized_pnl = np.zeros(self.crypto_num, dtype=np.float32)
        self.realized_pnl = 0.0
        self.total_asset = self.cash
        self.episode_return = 0.0
        self.gamma_return = 0.0
        
        self.returns_history = []
        self.liquidation_count = 0
        self.winning_trades = 0
        self.losing_trades = 0
        
        self.history = [{
            'time': self.time, 'cash': self.cash, 'total_asset': self.total_asset,
            'positions': self.positions.copy(), 'prices': self.current_price.copy(),
            'action': None, 'liquidated': [], 'trailing_stopped': [], 'reward': 0,
        }]
        
        return self.get_state()
    
    def _calculate_unrealized_pnl(self):
        pnl = np.zeros(self.crypto_num, dtype=np.float32)
        for i in range(self.crypto_num):
            if self.positions[i] != 0 and self.entry_prices[i] > 0:
                price_change = (self.current_price[i] - self.entry_prices[i]) / self.entry_prices[i]
                if self.positions[i] > 0:
                    pnl[i] = abs(self.positions[i]) * price_change
                else:
                    pnl[i] = abs(self.positions[i]) * (-price_change)
        return pnl
    
    def _check_liquidation(self):
        liquidated = []
        for i in range(self.crypto_num):
            if self.positions[i] != 0:
                position_value = abs(self.positions[i])
                margin_used = position_value / self.leverage
                
                if self.positions[i] > 0:
                    loss_pct = (self.entry_prices[i] - self.current_price[i]) / self.entry_prices[i]
                else:
                    loss_pct = (self.current_price[i] - self.entry_prices[i]) / self.entry_prices[i]
                
                max_loss_pct = (1 / self.leverage) - self.maintenance_margin
                
                if loss_pct >= max_loss_pct:
                    self.cash -= margin_used
                    self.realized_pnl -= margin_used
                    self.liquidation_count += 1
                    self.losing_trades += 1
                    
                    liquidated.append({
                        'coin': i, 'position': self.positions[i],
                        'entry_price': self.entry_prices[i],
                        'liquidation_price': self.current_price[i],
                        'loss': margin_used
                    })
                    
                    self.positions[i] = 0
                    self.entry_prices[i] = 0
                    self.highest_prices[i] = 0
                    self.lowest_prices[i] = np.inf
        return liquidated
    
    def _check_trailing_stop(self):
        if not self.use_trailing_stop:
            return []
        
        stopped = []
        for i in range(self.crypto_num):
            if self.positions[i] != 0:
                triggered = False
                if self.positions[i] > 0:
                    self.highest_prices[i] = max(self.highest_prices[i], self.current_price[i])
                    drop_pct = (self.highest_prices[i] - self.current_price[i]) / self.highest_prices[i]
                    if drop_pct >= self.trailing_stop_pct:
                        triggered = True
                else:
                    self.lowest_prices[i] = min(self.lowest_prices[i], self.current_price[i])
                    rise_pct = (self.current_price[i] - self.lowest_prices[i]) / self.lowest_prices[i]
                    if rise_pct >= self.trailing_stop_pct:
                        triggered = True
                
                if triggered:
                    pnl = self._close_position(i)
                    stopped.append({'coin': i, 'position': self.positions[i], 'pnl': pnl})
        return stopped
    
    def _close_position(self, coin_idx):
        if self.positions[coin_idx] == 0:
            return 0.0
        
        position_value = abs(self.positions[coin_idx])
        margin_used = position_value / self.leverage
        
        if self.positions[coin_idx] > 0:
            price_change = (self.current_price[coin_idx] - self.entry_prices[coin_idx]) / self.entry_prices[coin_idx]
        else:
            price_change = (self.entry_prices[coin_idx] - self.current_price[coin_idx]) / self.entry_prices[coin_idx]
        
        pnl = position_value * price_change - position_value * self.taker_fee
        self.cash += margin_used + pnl
        self.realized_pnl += pnl
        
        if pnl > 0:
            self.winning_trades += 1
        else:
            self.losing_trades += 1
        
        self.positions[coin_idx] = 0
        self.entry_prices[coin_idx] = 0
        self.highest_prices[coin_idx] = 0
        self.lowest_prices[coin_idx] = np.inf
        return pnl
    
    def _open_position(self, coin_idx, action):
        if action == 0:
            return
        
        available_capital = self.cash * 0.95
        margin_to_use = available_capital * abs(action) / self.crypto_num
        
        if margin_to_use < 100:
            return
        
        position_value = margin_to_use * self.leverage
        fee = position_value * self.taker_fee
        self.cash -= (margin_to_use + fee)
        
        self.positions[coin_idx] = position_value if action > 0 else -position_value
        self.entry_prices[coin_idx] = self.current_price[coin_idx]
        self.highest_prices[coin_idx] = self.current_price[coin_idx]
        self.lowest_prices[coin_idx] = self.current_price[coin_idx]
    
    def _calculate_reward(self, prev_total_asset, liquidated, trailing_stopped):
        # PnL reward
        pnl_change = self.total_asset - prev_total_asset
        pnl_reward = pnl_change * self.reward_scaling
        
        # Liquidation penalty
        liq_penalty = sum(-l['loss'] * self.reward_scaling * self.liquidation_penalty for l in liquidated)
        
        # Profit bonus
        profit_bonus = sum(s['pnl'] * self.reward_scaling * 0.5 for s in trailing_stopped if s['pnl'] > 0)
        
        # Risk penalty
        self.returns_history.append(pnl_change / prev_total_asset if prev_total_asset > 0 else 0)
        risk_penalty = 0
        if len(self.returns_history) > 10:
            vol = np.std(self.returns_history[-10:])
            if vol > 0.01:
                risk_penalty = -vol * 100 * self.reward_scaling
        
        return pnl_reward + liq_penalty + profit_bonus + risk_penalty
    
    def step(self, actions):
        self.time += 1
        self.current_price = self.price_array[self.time]
        self.current_tech = self.tech_array[self.time]
        prev_total_asset = self.total_asset
        
        liquidated = self._check_liquidation()
        trailing_stopped = self._check_trailing_stop()
        
        for i in range(self.crypto_num):
            action = actions[i]
            if any(l['coin'] == i for l in liquidated) or any(s['coin'] == i for s in trailing_stopped):
                continue
            
            if self.positions[i] == 0:
                if abs(action) > 0.1:
                    self._open_position(i, action)
            else:
                if self.positions[i] > 0:
                    if action < -0.1:
                        self._close_position(i)
                        if abs(action) > 0.3:
                            self._open_position(i, action)
                    elif action < 0.1:
                        self._close_position(i)
                else:
                    if action > 0.1:
                        self._close_position(i)
                        if abs(action) > 0.3:
                            self._open_position(i, action)
                    elif action > -0.1:
                        self._close_position(i)
        
        self.unrealized_pnl = self._calculate_unrealized_pnl()
        margin_in_pos = sum(abs(pos) / self.leverage for pos in self.positions if pos != 0)
        self.total_asset = self.cash + margin_in_pos + self.unrealized_pnl.sum()
        
        reward = self._calculate_reward(prev_total_asset, liquidated, trailing_stopped)
        done = self.time == self.max_step or self.total_asset < self.initial_capital * 0.1
        
        if done:
            if self.total_asset < self.initial_capital * 0.1:
                reward -= 1.0
            elif self.total_asset > self.initial_capital:
                reward += (self.total_asset / self.initial_capital - 1) * 0.1
        
        self.gamma_return = self.gamma_return * self.gamma + reward
        if done:
            self.episode_return = self.total_asset / self.initial_capital
        
        self.history.append({
            'time': self.time, 'cash': self.cash, 'total_asset': self.total_asset,
            'positions': self.positions.copy(), 'prices': self.current_price.copy(),
            'unrealized_pnl': self.unrealized_pnl.copy(), 'action': actions.copy(),
            'liquidated': liquidated, 'trailing_stopped': trailing_stopped, 'reward': reward,
        })
        
        return self.get_state(), reward, done, None
    
    def get_state(self):
        cash_ratio = self.cash / self.initial_capital
        pos_ratios = self.positions / (self.initial_capital * self.leverage) * 100
        pnl_ratios = self.unrealized_pnl / self.initial_capital * 100
        
        state = np.hstack([[cash_ratio], pos_ratios, pnl_ratios])
        for i in range(self.lookback):
            tech_i = self.tech_array[self.time - i] * 2**-15
            state = np.hstack((state, tech_i))
        return state.astype(np.float32)
    
    def close(self):
        pass

print("CryptoLeverageEnv defined!")

CryptoLeverageEnv defined!


In [4]:
class CryptoLeverageGym(gym.Env):
    """Gymnasium wrapper"""
    def __init__(self, config, **kwargs):
        super().__init__()
        self.env = CryptoLeverageEnv(config, **kwargs)
        self.observation_space = spaces.Box(-np.inf, np.inf, (self.env.state_dim,), np.float32)
        self.action_space = spaces.Box(-1, 1, (self.env.action_dim,), np.float32)
    
    def reset(self, seed=None, options=None):
        return self.env.reset(seed=seed, options=options), {}
    
    def step(self, action):
        state, reward, done, info = self.env.step(action)
        return state, reward, done, False, info or {}

# Config
train_config = {"price_array": train_price, "tech_array": train_tech}
test_config = {"price_array": test_price, "tech_array": test_tech}

env_kwargs = {
    'lookback': 1,
    'initial_capital': 1_000_000,
    'leverage': 50,
    'taker_fee': 0.0004,
    'trailing_stop_pct': 0.02,
    'use_trailing_stop': True,
    'reward_scaling': 1e-4,
    'liquidation_penalty': 10.0,
}

# Test environment
env_test = CryptoLeverageGym(config=train_config, **env_kwargs)
print(f"State Dim: {env_test.env.state_dim}")
print(f"Action Dim: {env_test.env.action_dim}")
print(f"Max Steps: {env_test.env.max_step}")

State Dim: 51
Action Dim: 5
Max Steps: 6821


# Part 4. Training (필수!)

**중요**: 기존 모델은 41차원, 새 환경은 51차원이므로 **반드시 새로 학습해야 합니다!**

In [5]:
# Training callback
class ProgressCallback(BaseCallback):
    def __init__(self, check_freq=1000):
        super().__init__()
        self.check_freq = check_freq
        
    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            env = self.training_env.envs[0]
            if hasattr(env, 'env'):
                asset = env.env.total_asset
                ret = (asset / env.env.initial_capital - 1) * 100
                liq = env.env.liquidation_count
                print(f"Step {self.n_calls}: Asset=${asset:,.0f}, Return={ret:.1f}%, Liquidations={liq}")
        return True

In [6]:
# Training parameters
TOTAL_TIMESTEPS = 50_000  # 빠른 테스트용. 더 나은 결과를 원하면 100_000~500_000

trained_models = {}

In [7]:
# Train PPO
print("="*50)
print("Training PPO for 50x Leverage Environment...")
print("="*50)

env_ppo = CryptoLeverageGym(config=train_config, **env_kwargs)

model_ppo = PPO(
    "MlpPolicy",
    env_ppo,
    verbose=0,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    clip_range=0.2,
    ent_coef=0.01,
)

model_ppo.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=ProgressCallback(check_freq=5000),
    progress_bar=True
)

model_ppo.save(f"{LEVERAGE_MODEL_DIR}/leverage_ppo")
trained_models['PPO'] = model_ppo
print("\nPPO training complete!")

Training PPO for 50x Leverage Environment...


ValueError: could not broadcast input array from shape (46,) into shape (51,)

In [ ]:
# Train A2C
print("="*50)
print("Training A2C for 50x Leverage Environment...")
print("="*50)

env_a2c = CryptoLeverageGym(config=train_config, **env_kwargs)

model_a2c = A2C(
    "MlpPolicy",
    env_a2c,
    verbose=0,
    learning_rate=7e-4,
    n_steps=5,
    gamma=0.99,
    ent_coef=0.01,
)

model_a2c.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=ProgressCallback(check_freq=5000),
    progress_bar=True
)

model_a2c.save(f"{LEVERAGE_MODEL_DIR}/leverage_a2c")
trained_models['A2C'] = model_a2c
print("\nA2C training complete!")

In [ ]:
print(f"\nTrained models: {list(trained_models.keys())}")
print(f"Saved to: {LEVERAGE_MODEL_DIR}")

# Part 5. Backtest

In [ ]:
def run_backtest(model, env_config, model_name, **env_kwargs):
    env = CryptoLeverageGym(config=env_config, **env_kwargs)
    obs, _ = env.reset()
    done = False
    step = 0
    
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        step += 1
        if step % 500 == 0:
            print(f"{model_name} Step {step}: ${env.env.total_asset:,.0f}")
    
    history = env.env.history
    df = pd.DataFrame({
        'step': [h['time'] for h in history],
        'account_value': [h['total_asset'] for h in history],
    })
    
    final_ret = (history[-1]['total_asset'] / history[0]['total_asset'] - 1) * 100
    total_liq = sum(len(h['liquidated']) for h in history)
    total_stop = sum(len(h['trailing_stopped']) for h in history)
    
    print(f"\n{model_name} Results:")
    print(f"  Return: {final_ret:.2f}%")
    print(f"  Final: ${history[-1]['total_asset']:,.0f}")
    print(f"  Liquidations: {total_liq}")
    print(f"  Trailing Stops: {total_stop}")
    print(f"  Win Rate: {env.env.winning_trades}/{env.env.winning_trades+env.env.losing_trades}")
    
    return df, history

In [ ]:
# Run backtest on TEST data
results = {}

for name, model in trained_models.items():
    print(f"\n{'='*50}")
    print(f"{name} Backtest (50x Leverage) - TEST DATA")
    print(f"{'='*50}")
    
    df, history = run_backtest(model, test_config, name, **env_kwargs)
    results[name] = {'account': df, 'history': history}

# Part 6. Visualization

In [ ]:
if results:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    colors = {'PPO': 'green', 'A2C': 'blue', 'SAC': 'orange'}
    
    # Portfolio value
    ax1 = axes[0]
    for name, data in results.items():
        ax1.plot(data['account']['step'], data['account']['account_value'],
                 label=name, color=colors.get(name, 'gray'), linewidth=2)
        
        # Mark liquidations
        for h in data['history']:
            for liq in h['liquidated']:
                ax1.scatter(h['time'], h['total_asset'], marker='x', color='red', s=100, zorder=5)
    
    ax1.axhline(y=1_000_000, color='gray', linestyle='--', alpha=0.5, label='Initial')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Portfolio Value ($)')
    ax1.set_title('50x Leverage Trading - Portfolio Value (X = Liquidation)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Crypto prices
    ax2 = axes[1]
    for i, pair in enumerate(crypto_pairs):
        prices = test_price[:, i]
        norm = prices / prices[0] * 100
        ax2.plot(norm, label=pair.replace('/USDT', ''), alpha=0.8)
    
    ax2.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Normalized Price')
    ax2.set_title('Crypto Price Movement')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Summary statistics
if results:
    summary = []
    for name, data in results.items():
        account = data['account']['account_value']
        history = data['history']
        
        total_return = (account.iloc[-1] / account.iloc[0] - 1) * 100
        peak = account.expanding().max()
        max_dd = ((account - peak) / peak * 100).min()
        
        total_liq = sum(len(h['liquidated']) for h in history)
        total_stop = sum(len(h['trailing_stopped']) for h in history)
        
        summary.append({
            'Model': name,
            'Final ($)': f"{account.iloc[-1]:,.0f}",
            'Return (%)': f"{total_return:.2f}",
            'Max DD (%)': f"{max_dd:.2f}",
            'Liquidations': total_liq,
            'Stops': total_stop,
        })
    
    print("\n" + "="*70)
    print("50x Leverage Backtest Summary")
    print("="*70)
    print(pd.DataFrame(summary).to_string(index=False))

# Part 7. Load Pre-trained Models (선택)

다음에 다시 실행할 때는 학습 없이 저장된 모델을 로드할 수 있습니다.

In [ ]:
# 저장된 모델 로드 (다음 실행 시 사용)
# trained_models = {}
# 
# try:
#     trained_models['PPO'] = PPO.load(f"{LEVERAGE_MODEL_DIR}/leverage_ppo")
#     print("PPO loaded")
# except Exception as e:
#     print(f"PPO not found: {e}")
# 
# try:
#     trained_models['A2C'] = A2C.load(f"{LEVERAGE_MODEL_DIR}/leverage_a2c")
#     print("A2C loaded")
# except Exception as e:
#     print(f"A2C not found: {e}")

print("Uncomment above code to load saved models")